# BERTScore — Ground Truth × Gherkin Generations (JSON)

Notebook adapted to the same data format used in the **Manhattan Distance** calculation:

- **Ground truth**: JSON containing `cases`, where each case has `case_id`, `original_case`, `reference_id`, and `gherkin`.
- **Generations**: JSON containing the metadata `model`, `technique`, `number_of_executions` and, for each case, a `generations` list.
- The association between reference and generation is performed by **`case_id`**, not by the position of the case in the file.
- The calculation preserves the configuration of the previous BERTScore notebook: `lang='pt'` and `rescale_with_baseline=True`.
- The notebook accepts **1 ground truth file and 1 or more generation files**.
- Precision, Recall, and F1 are exported as **percentages (0–100)**; each case is ranked by the **highest F1**.
- The calculation is performed in batches to avoid one BERTScore call for each individual pair.
- At the end, a **detailed CSV** containing all comparisons and rankings is generated.

> **Interpretation:** the higher the BERTScore F1, the greater the semantic similarity between the reference scenario and the generated scenario.


In [ ]:
# ============================================================
# 1. DEPENDENCY INSTALLATION
# ============================================================

!pip -q install bert-score


In [ ]:
# ============================================================
# 2. IMPORTS AND CONFIGURATION
# ============================================================

import json
import re
import warnings

import pandas as pd
import torch
from IPython.display import display
from bert_score import score as bertscore_score

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", None)


BERTSCORE_LANG = "pt"
RESCALE_WITH_BASELINE = True

# The notebook uses the GPU when available and falls back to the CPU otherwise.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Adjust this value if GPU/CPU memory is limited.
BERTSCORE_BATCH_SIZE = 16 if DEVICE == "cuda" else 8

# If True, the CSV will be downloaded automatically at the end in Google Colab.
AUTO_DOWNLOAD_CSV = True

print(f"Selected device: {DEVICE}")
print(f"Batch size: {BERTSCORE_BATCH_SIZE}")


In [ ]:
# ============================================================
# 3. UPLOAD AND AUTOMATIC JSON IDENTIFICATION
# ============================================================

def load_json_bytes(file_name, content):
    # Accepts UTF-8 with or without BOM.
    try:
        text = content.decode("utf-8-sig")
        return json.loads(text)
    except Exception as error:
        raise ValueError(
            f"Could not read '{file_name}' as JSON: {error}"
        ) from error


def classify_json(file_name, data):
    # Classifies the file as ground_truth, generations, or unknown.
    if not isinstance(data, dict):
        return "unknown"

    cases = data.get("cases")
    if not isinstance(cases, list):
        return "unknown"

    if data.get("is_reference_base") is True:
        return "ground_truth"

    if cases:
        first_case = cases[0]
        if isinstance(first_case, dict):
            if "generations" in first_case:
                return "generations"
            if "reference_id" in first_case and "gherkin" in first_case:
                return "ground_truth"

    return "unknown"


def process_upload(uploaded):
    uploaded_files = []

    for file_name, content in uploaded.items():
        if not file_name.lower().endswith(".json"):
            print(f"⚠ Ignored (not JSON): {file_name}")
            continue

        data = load_json_bytes(file_name, content)
        file_type = classify_json(file_name, data)
        uploaded_files.append({
            "name": file_name,
            "type": file_type,
            "data": data
        })

    return uploaded_files


try:
    from google.colab import files
except ImportError as error:
    raise RuntimeError(
        "This notebook was prepared for interactive upload in Google Colab. "
        "Run it in Colab or adapt this cell for local file reading."
    ) from error


# ------------------------------------------------------------
# STEP 1 — Ground truth
# ------------------------------------------------------------
print("STEP 1/2 — Upload the ground truth JSON file:")
ground_truth_upload = files.upload()
json_files = process_upload(ground_truth_upload)

ground_truths = [
    item for item in json_files if item["type"] == "ground_truth"
]
generation_files = [
    item for item in json_files if item["type"] == "generations"
]
unknown_files = [
    item["name"] for item in json_files if item["type"] == "unknown"
]

if unknown_files:
    print("⚠ JSON file(s) with unrecognized structure:", unknown_files)

if len(ground_truths) != 1:
    raise ValueError(
        f"Exactly 1 ground truth file is required. "
        f"{len(ground_truths)} were identified. "
        "Check whether the file contains 'is_reference_base': true "
        "or cases with 'reference_id' and 'gherkin'."
    )

ground_truth_file_name = ground_truths[0]["name"]
ground_truth = ground_truths[0]["data"]

print(f"\n✓ Ground truth identified: {ground_truth_file_name}")


# ------------------------------------------------------------
# STEP 2 — Generations
# ------------------------------------------------------------
# If generations were uploaded together with the ground truth, reuse them.
# Otherwise, open a second upload selector.
if not generation_files:
    print("\nSTEP 2/2 — Now upload one or more generation JSON files:")
    generation_upload = files.upload()
    new_files = process_upload(generation_upload)

    new_ground_truths = [
        item for item in new_files if item["type"] == "ground_truth"
    ]
    if new_ground_truths:
        print(
            "⚠ Additional ground truth ignored during the generation step:",
            [item["name"] for item in new_ground_truths]
        )

    new_unknown_files = [
        item["name"] for item in new_files if item["type"] == "unknown"
    ]
    if new_unknown_files:
        print(
            "⚠ JSON file(s) with unrecognized structure:",
            new_unknown_files
        )

    generation_files.extend(
        item for item in new_files if item["type"] == "generations"
    )

if not generation_files:
    raise ValueError(
        "No generation file was identified. Generation files must contain "
        "'cases' and, within each case, the 'generations' key."
    )

print(f"\n✓ Generation files identified: {len(generation_files)}")
for generation_file in generation_files:
    data = generation_file["data"]
    print(
        f"  - {generation_file['name']} | model={data.get('model')} | "
        f"technique={data.get('technique')} | "
        f"declared executions={data.get('number_of_executions')}"
    )


In [ ]:
# ============================================================
# 4. DATA VALIDATION
# ============================================================

def index_ground_truth(ground_truth_data):
    references = {}
    duplicates = []

    for case in ground_truth_data.get("cases", []):
        case_id = case.get("case_id")

        if not case_id:
            continue

        if case_id in references:
            duplicates.append(case_id)

        references[case_id] = case

    if duplicates:
        raise ValueError(
            f"Duplicate case_id value(s) in the ground truth: "
            f"{sorted(set(duplicates))}"
        )

    return references


references = index_ground_truth(ground_truth)
validation_warnings = []

for generation_file in generation_files:
    file_name = generation_file["name"]
    data = generation_file["data"]
    file_case_ids = []
    declared_executions = data.get("number_of_executions")

    for case in data.get("cases", []):
        case_id = case.get("case_id")
        file_case_ids.append(case_id)

        if case_id not in references:
            validation_warnings.append(
                f"{file_name}: {case_id} exists in the generations, "
                "but not in the ground truth."
            )
            continue

        reference_original_case = references[case_id].get("original_case")
        generated_original_case = case.get("original_case")

        if (
            reference_original_case is not None
            and generated_original_case is not None
            and reference_original_case != generated_original_case
        ):
            validation_warnings.append(
                f"{file_name}: divergent original_case in {case_id}."
            )

        generations = case.get("generations", [])

        if (
            declared_executions is not None
            and len(generations) != declared_executions
        ):
            validation_warnings.append(
                f"{file_name}: {case_id} contains {len(generations)} generations, "
                f"but the file declares {declared_executions}."
            )

        executions = [
            generation.get("execution") for generation in generations
        ]
        valid_executions = [
            execution for execution in executions if execution is not None
        ]

        if len(valid_executions) != len(set(valid_executions)):
            validation_warnings.append(
                f"{file_name}: duplicate execution numbers in {case_id}."
            )

    reference_ids = set(references)
    generation_ids = set(file_case_ids)

    missing_case_ids = sorted(reference_ids - generation_ids)
    if missing_case_ids:
        validation_warnings.append(
            f"{file_name}: {len(missing_case_ids)} ground-truth case_id value(s) "
            "do not appear in the generations. "
            f"Examples: {missing_case_ids[:10]}"
        )

print(f"Cases in ground truth: {len(references)}")

if validation_warnings:
    print(
        f"\n⚠ {len(validation_warnings)} validation warning(s) were found:"
    )
    for warning in validation_warnings:
        print(" -", warning)
else:
    print("\n✓ Structure validated without warnings.")


In [ ]:
# ============================================================
# 5. COMPARISON PREPARATION
# ============================================================

comparisons = []

for generation_file in generation_files:
    generations_file_name = generation_file["name"]
    data = generation_file["data"]

    model = data.get("model", "")
    technique = data.get("technique", "")
    declared_executions = data.get("number_of_executions")

    for generated_case in data.get("cases", []):
        case_id = generated_case.get("case_id")
        reference = references.get(case_id)

        if reference is None:
            continue

        reference_gherkin = reference.get("gherkin", "")

        for generation in generated_case.get("generations", []):
            generated_gherkin = generation.get("gherkin", "")

            comparisons.append({
                # Compatibility contract: output field names remain unchanged.
                "arquivo_ground_truth": ground_truth_file_name,
                "arquivo_geracoes": generations_file_name,
                "modelo": model,
                "tecnica": technique,
                "execucoes_declaradas": declared_executions,
                "case_id": case_id,
                "source_id": reference.get("source_id"),
                "source_line": reference.get("source_line"),
                "original_case": reference.get("original_case"),
                "reference_id": reference.get("reference_id"),
                "generation_id": generation.get("generation_id"),
                "execucao": generation.get("execution"),
                "gherkin_ground_truth": (
                    "" if reference_gherkin is None else str(reference_gherkin)
                ),
                "gherkin_gerado": (
                    "" if generated_gherkin is None else str(generated_gherkin)
                ),
            })

if not comparisons:
    raise ValueError("No comparison could be prepared.")

print(f"✓ Comparisons prepared: {len(comparisons):,}")


In [ ]:
# ============================================================
# 6. BERTSCORE CALCULATION
# ============================================================

# In BERTScore:
#   cands = generated texts
#   refs  = reference texts
#
# Keep lang='pt' and rescale_with_baseline=True exactly as in the original.
candidates = [row["gherkin_gerado"] for row in comparisons]
reference_texts = [
    row["gherkin_ground_truth"] for row in comparisons
]

print(
    f"Calculating BERTScore for {len(comparisons):,} pairs "
    f"on {DEVICE.upper()}..."
)

P, R, F1 = bertscore_score(
    candidates,
    reference_texts,
    lang=BERTSCORE_LANG,
    rescale_with_baseline=RESCALE_WITH_BASELINE,
    batch_size=BERTSCORE_BATCH_SIZE,
    device=DEVICE,
    verbose=True,
)

precision_percentages = (
    P.detach().cpu().numpy() * 100
).tolist()
recall_percentages = (
    R.detach().cpu().numpy() * 100
).tolist()
f1_percentages = (
    F1.detach().cpu().numpy() * 100
).tolist()

results = []

for record, precision, recall, f1 in zip(
    comparisons,
    precision_percentages,
    recall_percentages,
    f1_percentages
):
    row = dict(record)

    # Preserve the original percentage convention while retaining more
    # decimal places in the CSV to avoid losing statistical precision.
    row["bertscore_precision"] = float(precision)
    row["bertscore_recall"] = float(recall)
    row["bertscore_f1"] = float(f1)

    results.append(row)

results_df = pd.DataFrame(results)

# Compatibility contract: these column names remain unchanged.
ranking_keys = [
    "arquivo_geracoes",
    "modelo",
    "tecnica",
    "case_id"
]

# In BERTScore, higher F1 = better result.
results_df = results_df.sort_values(
    ranking_keys + ["bertscore_f1", "execucao"],
    ascending=[True, True, True, True, False, True],
    kind="stable",
    na_position="last"
).reset_index(drop=True)

# Sequential ranking after sorting by highest F1.
# As in the Manhattan notebook, ties occupy successive positions.
results_df["ranking_no_caso"] = (
    results_df
    .groupby(ranking_keys, dropna=False)
    .cumcount() + 1
)

# Compatibility contract: exported CSV columns remain exactly as in the original.
output_columns = [
    "modelo",
    "tecnica",
    "case_id",
    "source_id",
    "original_case",
    "execucao",
    "bertscore_precision",
    "bertscore_recall",
    "bertscore_f1",
    "ranking_no_caso",
    "generation_id",
    "reference_id",
    "gherkin_ground_truth",
    "gherkin_gerado",
    "execucoes_declaradas",
    "arquivo_ground_truth",
    "arquivo_geracoes",
]

results_df = results_df[output_columns]

print(f"\n✓ Comparisons calculated: {len(results_df):,}")
print(f"✓ Cases evaluated: {results_df['case_id'].nunique():,}")


In [ ]:
# ============================================================
# 7. ORGANIZED TABLES
# ============================================================

overall_summary_df = (
    results_df
    .groupby(["arquivo_geracoes", "modelo", "tecnica"], dropna=False)
    .agg(
        casos=("case_id", "nunique"),
        comparacoes=("bertscore_f1", "count"),
        f1_media=("bertscore_f1", "mean"),
        f1_mediana=("bertscore_f1", "median"),
        desvio_padrao=("bertscore_f1", "std"),
        f1_minima=("bertscore_f1", "min"),
        f1_maxima=("bertscore_f1", "max"),
    )
    .reset_index()
)

print("OVERALL SUMMARY")
display(
    overall_summary_df.style.format({
        "f1_media": "{:.2f}",
        "f1_mediana": "{:.2f}",
        "desvio_padrao": "{:.2f}",
        "f1_minima": "{:.2f}",
        "f1_maxima": "{:.2f}",
    })
)

case_summary_df = (
    results_df
    .groupby(
        [
            "arquivo_geracoes",
            "modelo",
            "tecnica",
            "case_id",
            "original_case"
        ],
        dropna=False
    )
    .agg(
        execucoes_avaliadas=("execucao", "count"),
        f1_media=("bertscore_f1", "mean"),
        f1_mediana=("bertscore_f1", "median"),
        desvio_padrao=("bertscore_f1", "std"),
        melhor_f1=("bertscore_f1", "max"),
        pior_f1=("bertscore_f1", "min"),
    )
    .reset_index()
    .sort_values(["modelo", "tecnica", "case_id"])
)

print("\nSUMMARY BY CASE — first 30 rows")
display(
    case_summary_df.head(30).style.format({
        "f1_media": "{:.2f}",
        "f1_mediana": "{:.2f}",
        "desvio_padrao": "{:.2f}",
        "melhor_f1": "{:.2f}",
        "pior_f1": "{:.2f}",
    })
)

print("\nDETAILED COMPARISONS — first 50 rows")
display_columns = [
    "modelo",
    "tecnica",
    "case_id",
    "original_case",
    "execucao",
    "bertscore_precision",
    "bertscore_recall",
    "bertscore_f1",
    "ranking_no_caso",
]

display(
    results_df[display_columns]
    .head(50)
    .style
    .format({
        "bertscore_precision": "{:.2f}",
        "bertscore_recall": "{:.2f}",
        "bertscore_f1": "{:.2f}",
    })
)


In [ ]:
# ============================================================
# 8. QUICK CASE LOOKUP
# ============================================================

def view_case(case_id):
    # Displays all executions of a case_id from highest to lowest F1.
    case_results = results_df[
        results_df["case_id"] == case_id
    ].copy()

    if case_results.empty:
        print(f"No result found for {case_id}.")
        return

    # Compatibility contract: column labels remain unchanged.
    columns = [
        "modelo",
        "tecnica",
        "case_id",
        "original_case",
        "execucao",
        "bertscore_precision",
        "bertscore_recall",
        "bertscore_f1",
        "ranking_no_caso",
        "gherkin_ground_truth",
        "gherkin_gerado",
    ]

    display(
        case_results[columns]
        .sort_values(
            ["modelo", "tecnica", "bertscore_f1", "execucao"],
            ascending=[True, True, False, True]
        )
        .style
        .format({
            "bertscore_precision": "{:.2f}",
            "bertscore_recall": "{:.2f}",
            "bertscore_f1": "{:.2f}",
        })
    )


first_case_id = results_df["case_id"].iloc[0]
print(f"Query example: {first_case_id}")
view_case(first_case_id)

# To query another case:
# view_case("TC_261")


In [ ]:
# ============================================================
# 9. CSV EXPORT
# ============================================================

def slug(text):
    text = str(text or "").strip().lower()
    text = re.sub(r"[^a-z0-9._-]+", "-", text)
    text = re.sub(r"-+", "-", text).strip("-")
    return text or "sem-identificacao"


unique_metadata = (
    results_df[["modelo", "tecnica"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

if len(unique_metadata) == 1:
    model = unique_metadata.loc[0, "modelo"]
    technique = unique_metadata.loc[0, "tecnica"]

    # Compatibility contract: keep the original output filename pattern.
    csv_file_name = (
        f"metricas_bertscore_{slug(model)}_{slug(technique)}.csv"
    )
else:
    csv_file_name = (
        "metricas_bertscore_multiplos_modelos_tecnicas.csv"
    )

# UTF-8 with BOM facilitates correct handling of accents in Excel.
results_df.to_csv(
    csv_file_name,
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ CSV generated: {csv_file_name}")
print(f"✓ Rows exported: {len(results_df):,}")

if AUTO_DOWNLOAD_CSV:
    try:
        from google.colab import files
        files.download(csv_file_name)
    except Exception as error:
        print(f"Automatic download was not completed: {error}")
